# Fixed eight-quantity models: outer domains and logged ratios

Prepared 17 October 2026. Extensive fitting is **not started** by this notebook. We compare four fixed input specifications, each fitted with Ridge, Elastic Net, random forest and gradient boosting. We do not search for different variable subsets or choose different model families for individual held-out networks.

The eight inputs are $R_{\mu_L}$, either $R_{\sigma_L}$ or $R_{\mathrm{CV}_L}$, $R_{\bar W}$, $R_{\mathrm{CV}_W}$, parent dominance $\bar d_p$, parent inlet exposure $r_{\mathrm{in},p}$, $R_H$, and $-\log_{10}(K_c/K_p)$ using the agreed $\alpha=1$ resistance law. $R_X=X_c/X_p$.

All geometric measurements use the outer domain: the full containing compound for a member edit, the closed standalone unit otherwise, and the network for a centerline. Parent and child describe the same edit. Exposure is measured from the parent network to that domain inlet.

## Raw versus logged

1. SD spread, raw ratios.
2. SD spread, $\ln(1+R)$ ratios.
3. CV spread, raw ratios.
4. CV spread, $\ln(1+R)$ ratios.

Only the five geometric ratios are transformed. Dominance and exposure remain raw; resistance already has its agreed negative base-10 log ratio. Zero ratios are physically valid when a collapse eliminates spread or branching, so ordinary $\ln R$ is undefined. We use natural **log1p**, without epsilon floors or excluding complete-removal cases. Unchanged geometry ($R=1$) maps to $\ln2$; training-fold centering handles the offset. This differs from a signed log ratio. No response is log-transformed.

There are 16 fixed model/input specifications and 96 hyperparameter configurations. Each model has six settings. Monotonic transforms do not create new information and may produce identical or nearly identical tree results. The experiment tests whether the transformed relationship helps the chosen estimator.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'RAPID/src/rapid_tools').is_dir())
OUT=ROOT/'RAPID/outputs/hydrograph_focused_models_20261017'
design=json.loads((OUT/'design.json').read_text())
def table(df):
    lines=['| '+' | '.join(map(str,df.columns))+' |','| '+' | '.join(['---']*len(df.columns))+' |']
    for row in df.itertuples(index=False,name=None):lines.append('| '+' | '.join(str(x).replace('|',' / ') for x in row)+' |')
    display(Markdown('\n'.join(lines)))
table(pd.read_csv(OUT/'input_audit.csv'))
table(pd.read_csv(OUT/'exclusions.csv'))
table(pd.read_csv(OUT/'run_plan.csv'))

| feature | symbol | n | zero | unique | minimum | maximum |
| --- | --- | --- | --- | --- | --- | --- |
| B_ratio_route_length_mean | $R_{\mu_L}$ | 2185 | 0 | 2183 | 0.58586326854475 | 1.1268130091423771 |
| B_ratio_route_length_sd | $R_{\sigma_L}$ | 2185 | 451 | 1734 | 0.0 | 15.456428197096708 |
| B_ratio_route_length_cv | $R_{\mathrm{CV}_L}$ | 2185 | 451 | 1734 | 0.0 | 16.270147881031253 |
| B_ratio_width_mean | $R_{\bar W}$ | 2185 | 0 | 2185 | 0.9700437124399368 | 7.452604861405111 |
| B_ratio_width_cv | $R_{\mathrm{CV}_W}$ | 2185 | 412 | 1774 | 0.0 | 2.205218682133523 |
| B_ratio_route_entropy | $R_H$ | 2185 | 451 | 1734 | 0.0 | 1.6300887406488604 |
| B_parent_dominance_mean | $\bar d_p$ | 2185 | 0 | 450 | 0.5011346264178344 | 0.959030028204866 |
| B_exposure | $r_{\mathrm{in},p}$ | 2185 | 0 | 12 | 0.9999999999999988 | 1.0000000000000002 |
| B_K_log_a1 | $-\log_{10}(K_c/K_p)$ | 2185 | 0 | 2185 | -2.5546681706534033 | 3.016227668148356 |

| example | state | reason |
| --- | --- | --- |
| 37 | island_37_098 | previous resistance/domain quality exclusion; B_K_log_a1: undefined/nonfinite |
| 41 | S061_unit_57 | previous resistance/domain quality exclusion; B_ratio_route_length_mean: undefined/nonfinite; B_ratio_route_length_sd: undefined/nonfinite; B_ratio_route_length_cv: undefined/nonfinite; B_ratio_width_mean: undefined/nonfinite; B_ratio_width_cv: undefined/nonfinite; B_ratio_route_entropy: undefined/nonfinite; B_parent_dominance_mean: undefined/nonfinite; B_exposure: undefined/nonfinite; B_K_log_a1: undefined/nonfinite |

| cohort | scenario | n | examples | fits_per_target |
| --- | --- | --- | --- | --- |
| partial | all | 2170 | 15 | 4560 |
| partial | without54 | 2089 | 14 | 4256 |
| including_centerlines | all | 2185 | 15 | 4560 |
| including_centerlines | without54 | 2103 | 14 | 4256 |

## Coverage and validation

The current common cohort has 2,185 comparisons: 2,170 partial collapses and 15 centerlines. The same two domain/resistance quality exclusions as the previous benchmark remain. All ratios and parent dominance are finite on this cohort; zero denominators do not currently cause further exclusions. Future undefined ratios are explicitly excluded from **every** specification, not filled using an arbitrary epsilon.

**Exposure equals 1 for all eligible outer domains.** It stays in the conceptual specification but drops as a constant during fitting, leaving seven varying inputs. This finding concerns these chosen domains, not a general claim that every closed compound has exposure 1.

We retain **leave-one-whole-example-out** validation for comparability with the previous results. Within each training set, three grouped folds select hyperparameters separately for each fixed specification. No broad selection across input lists/model families is reported. Balanced grouped validation remains a separate, unimplemented comparison.

Training uses equal total weight per example. Selection minimizes equal-example MSE; reports include MAE, RMSE and $R^2$, pair-weighted scores and per-example results. Scaling and constant/duplicate removal are training-only; no correlation pruning or relevance screening is used. Elastic Net can shrink coefficients to zero. Hyperparameters may differ between validation folds; after selecting the final model specification, final tuning/refitting produces one deployed model.

All/without54 are separately refitted. Centerlines are an optional separate cohort. All six hydrograph definitions remain unchanged, including global maximum timing in 54; no crest diagnostic is a predictor. Saved predictions are also assessed against 1% and 2% windows without refitting. Comparing the outer rankings is exploratory because we have already used these networks for development.

In [2]:
table(pd.DataFrame([{'specification':s['name'],'inputs':len(s['columns']),'settings':json.dumps(s['grid'])} for s in design['specifications']]))

| specification | inputs | settings |
| --- | --- | --- |
| sd__raw__ridge | 8 | [{"alpha": 0.01}, {"alpha": 0.1}, {"alpha": 1.0}, {"alpha": 10.0}, {"alpha": 100.0}, {"alpha": 1000.0}] |
| sd__raw__elastic_net | 8 | [{"alpha": 0.001, "l1_ratio": 0.2}, {"alpha": 0.001, "l1_ratio": 0.8}, {"alpha": 0.01, "l1_ratio": 0.2}, {"alpha": 0.01, "l1_ratio": 0.8}, {"alpha": 0.1, "l1_ratio": 0.2}, {"alpha": 0.1, "l1_ratio": 0.8}] |
| sd__raw__forest | 8 | [{"max_depth": 3, "min_samples_leaf": 5, "max_features": 1.0}, {"max_depth": 3, "min_samples_leaf": 15, "max_features": 1.0}, {"max_depth": 6, "min_samples_leaf": 5, "max_features": 1.0}, {"max_depth": 6, "min_samples_leaf": 15, "max_features": 1.0}, {"max_depth": null, "min_samples_leaf": 5, "max_features": 1.0}, {"max_depth": null, "min_samples_leaf": 15, "max_features": 1.0}] |
| sd__raw__boosting | 8 | [{"max_leaf_nodes": 7, "l2_regularization": 0.0, "learning_rate": 0.06}, {"max_leaf_nodes": 7, "l2_regularization": 1.0, "learning_rate": 0.06}, {"max_leaf_nodes": 7, "l2_regularization": 10.0, "learning_rate": 0.06}, {"max_leaf_nodes": 15, "l2_regularization": 0.0, "learning_rate": 0.06}, {"max_leaf_nodes": 15, "l2_regularization": 1.0, "learning_rate": 0.06}, {"max_leaf_nodes": 15, "l2_regularization": 10.0, "learning_rate": 0.06}] |
| sd__log1p__ridge | 8 | [{"alpha": 0.01}, {"alpha": 0.1}, {"alpha": 1.0}, {"alpha": 10.0}, {"alpha": 100.0}, {"alpha": 1000.0}] |
| sd__log1p__elastic_net | 8 | [{"alpha": 0.001, "l1_ratio": 0.2}, {"alpha": 0.001, "l1_ratio": 0.8}, {"alpha": 0.01, "l1_ratio": 0.2}, {"alpha": 0.01, "l1_ratio": 0.8}, {"alpha": 0.1, "l1_ratio": 0.2}, {"alpha": 0.1, "l1_ratio": 0.8}] |
| sd__log1p__forest | 8 | [{"max_depth": 3, "min_samples_leaf": 5, "max_features": 1.0}, {"max_depth": 3, "min_samples_leaf": 15, "max_features": 1.0}, {"max_depth": 6, "min_samples_leaf": 5, "max_features": 1.0}, {"max_depth": 6, "min_samples_leaf": 15, "max_features": 1.0}, {"max_depth": null, "min_samples_leaf": 5, "max_features": 1.0}, {"max_depth": null, "min_samples_leaf": 15, "max_features": 1.0}] |
| sd__log1p__boosting | 8 | [{"max_leaf_nodes": 7, "l2_regularization": 0.0, "learning_rate": 0.06}, {"max_leaf_nodes": 7, "l2_regularization": 1.0, "learning_rate": 0.06}, {"max_leaf_nodes": 7, "l2_regularization": 10.0, "learning_rate": 0.06}, {"max_leaf_nodes": 15, "l2_regularization": 0.0, "learning_rate": 0.06}, {"max_leaf_nodes": 15, "l2_regularization": 1.0, "learning_rate": 0.06}, {"max_leaf_nodes": 15, "l2_regularization": 10.0, "learning_rate": 0.06}] |
| cv__raw__ridge | 8 | [{"alpha": 0.01}, {"alpha": 0.1}, {"alpha": 1.0}, {"alpha": 10.0}, {"alpha": 100.0}, {"alpha": 1000.0}] |
| cv__raw__elastic_net | 8 | [{"alpha": 0.001, "l1_ratio": 0.2}, {"alpha": 0.001, "l1_ratio": 0.8}, {"alpha": 0.01, "l1_ratio": 0.2}, {"alpha": 0.01, "l1_ratio": 0.8}, {"alpha": 0.1, "l1_ratio": 0.2}, {"alpha": 0.1, "l1_ratio": 0.8}] |
| cv__raw__forest | 8 | [{"max_depth": 3, "min_samples_leaf": 5, "max_features": 1.0}, {"max_depth": 3, "min_samples_leaf": 15, "max_features": 1.0}, {"max_depth": 6, "min_samples_leaf": 5, "max_features": 1.0}, {"max_depth": 6, "min_samples_leaf": 15, "max_features": 1.0}, {"max_depth": null, "min_samples_leaf": 5, "max_features": 1.0}, {"max_depth": null, "min_samples_leaf": 15, "max_features": 1.0}] |
| cv__raw__boosting | 8 | [{"max_leaf_nodes": 7, "l2_regularization": 0.0, "learning_rate": 0.06}, {"max_leaf_nodes": 7, "l2_regularization": 1.0, "learning_rate": 0.06}, {"max_leaf_nodes": 7, "l2_regularization": 10.0, "learning_rate": 0.06}, {"max_leaf_nodes": 15, "l2_regularization": 0.0, "learning_rate": 0.06}, {"max_leaf_nodes": 15, "l2_regularization": 1.0, "learning_rate": 0.06}, {"max_leaf_nodes": 15, "l2_regularization": 10.0, "learning_rate": 0.06}] |
| cv__log1p__ridge | 8 | [{"alpha": 0.01}, {"alpha": 0.1}, {"alpha": 1.0}, {"alpha": 10.0}, {"alpha": 100.0}, {"alpha": 1000.0}] |
| cv__log1p__elastic_net | 8 | [{"alpha": 0.001, "l1_ratio": 0.2}, {"alpha": 0.001, "l1_ratio": 0.8}, {"alpha": 0.01, "l1_ratio": 0.2}, {"alpha": 0.01, "l1_ratio": 0.8}, {"alpha": 0.1, "l1_ratio": 0.2}, {"alpha": 0.1, "l1_ratio": 0.8}] |
| cv__log1p__forest | 8 | [{"max_depth": 3, "min_samples_leaf": 5, "max_features": 1.0}, {"max_depth": 3, "min_samples_leaf": 15, "max_features": 1.0}, {"max_depth": 6, "min_samples_leaf": 5, "max_features": 1.0}, {"max_depth": 6, "min_samples_leaf": 15, "max_features": 1.0}, {"max_depth": null, "min_samples_leaf": 5, "max_features": 1.0}, {"max_depth": null, "min_samples_leaf": 15, "max_features": 1.0}] |
| cv__log1p__boosting | 8 | [{"max_leaf_nodes": 7, "l2_regularization": 0.0, "learning_rate": 0.06}, {"max_leaf_nodes": 7, "l2_regularization": 1.0, "learning_rate": 0.06}, {"max_leaf_nodes": 7, "l2_regularization": 10.0, "learning_rate": 0.06}, {"max_leaf_nodes": 15, "l2_regularization": 0.0, "learning_rate": 0.06}, {"max_leaf_nodes": 15, "l2_regularization": 1.0, "learning_rate": 0.06}, {"max_leaf_nodes": 15, "l2_regularization": 10.0, "learning_rate": 0.06}] |

## Run in your terminal

From the repository root, this starts both primary responses on partial collapses, all/without54 (**17,632 fits**):

```bash
OMP_NUM_THREADS=1 OPENBLAS_NUM_THREADS=1 MKL_NUM_THREADS=1 \
/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python \
RAPID/investigations/hydrograph_predictors_20261014/focused_models.py \
--run --jobs 2
```

For a smaller first batch (**4,560 fits**), add options on the same command:

```bash
OMP_NUM_THREADS=1 OPENBLAS_NUM_THREADS=1 MKL_NUM_THREADS=1 \
/opt/anaconda3/envs/river-hierarchy-rivgraph/bin/python \
RAPID/investigations/hydrograph_predictors_20261014/focused_models.py \
--run --jobs 2 --targets peak_shift_hours --scenarios all
```

`--jobs` controls CPU workers within forests. Use `--cohorts including_centerlines` for the separate expanded analysis. Supporting targets can be passed to `--targets`: `centroid_shift_hours duration80_change_pct volume_change_pct asymmetry_change`.

Without `--run`, only preparation executes. Repeat a command to resume completed checkpoints. Do not run simultaneous writers into the same output directory. Changed inputs or code require a new `--output` directory once checkpoints exist. No production outputs or previous search checkpoints are modified.

In [3]:
COHORT='partial'
SCENARIO='all'
TARGET='peak_shift_hours'
if (OUT/'scores.csv').exists():
    scores=pd.read_csv(OUT/'scores.csv')
    z=scores[(scores.cohort==COHORT)&(scores.scenario==SCENARIO)&(scores.response==TARGET)&(scores.weighting=='equal_example')].copy()
    if TARGET.endswith('_hours'):z['MAE [min]']=60*z.mae
    table(z.sort_values('rmse').round(5))
else:
    print('Prepared only: no fitted-model scores yet.')

| cohort | scenario | response | pipeline | weighting | n | mae | rmse | r2 | MAE [min] |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| partial | all | peak_shift_hours | cv__raw__ridge | equal_example | 2170 | 0.11312 | 0.22952 | 0.23164 | 6.78729 |
| partial | all | peak_shift_hours | sd__raw__ridge | equal_example | 2170 | 0.11323 | 0.2296 | 0.23112 | 6.79402 |
| partial | all | peak_shift_hours | cv__log1p__elastic_net | equal_example | 2170 | 0.11214 | 0.22976 | 0.23002 | 6.72811 |
| partial | all | peak_shift_hours | sd__log1p__elastic_net | equal_example | 2170 | 0.11226 | 0.22982 | 0.22965 | 6.73546 |
| partial | all | peak_shift_hours | cv__log1p__ridge | equal_example | 2170 | 0.11375 | 0.23106 | 0.22131 | 6.82492 |
| partial | all | peak_shift_hours | sd__log1p__ridge | equal_example | 2170 | 0.11384 | 0.23107 | 0.2212 | 6.83059 |
| partial | all | peak_shift_hours | cv__raw__elastic_net | equal_example | 2170 | 0.11364 | 0.23138 | 0.2191 | 6.81847 |
| partial | all | peak_shift_hours | sd__raw__elastic_net | equal_example | 2170 | 0.11368 | 0.23139 | 0.21907 | 6.82061 |
| partial | all | peak_shift_hours | sd__log1p__forest | equal_example | 2170 | 0.11188 | 0.23561 | 0.19035 | 6.71305 |
| partial | all | peak_shift_hours | cv__log1p__forest | equal_example | 2170 | 0.11184 | 0.23561 | 0.19035 | 6.71043 |
| partial | all | peak_shift_hours | sd__raw__forest | equal_example | 2170 | 0.1119 | 0.23571 | 0.18962 | 6.71425 |
| partial | all | peak_shift_hours | cv__raw__forest | equal_example | 2170 | 0.11186 | 0.23571 | 0.18961 | 6.71156 |
| partial | all | peak_shift_hours | sd__raw__boosting | equal_example | 2170 | 0.11604 | 0.23725 | 0.17901 | 6.96232 |
| partial | all | peak_shift_hours | sd__log1p__boosting | equal_example | 2170 | 0.11499 | 0.23764 | 0.17633 | 6.89916 |
| partial | all | peak_shift_hours | cv__raw__boosting | equal_example | 2170 | 0.11547 | 0.23853 | 0.17013 | 6.92813 |
| partial | all | peak_shift_hours | cv__log1p__boosting | equal_example | 2170 | 0.11533 | 0.23869 | 0.16903 | 6.91993 |
| partial | all | peak_shift_hours | training_mean | equal_example | 2170 | 0.14014 | 0.26464 | -0.02146 | 8.40862 |
| partial | all | peak_shift_hours | zero | equal_example | 2170 | 0.13829 | 0.28857 | -0.21461 | 8.29729 |

In [4]:
if (OUT/'predictions.parquet').exists():
    import plotly.express as px
    pred=pd.read_parquet(OUT/'predictions.parquet')
    PIPELINE='sd__raw__ridge'
    z=pred[(pred.cohort==COHORT)&(pred.scenario==SCENARIO)&(pred.response==TARGET)&(pred.pipeline==PIPELINE)]
    if len(z):
        fig=px.scatter(z,x='observed',y='predicted',color='example',hover_data=['state'],template='plotly_white',title=PIPELINE+' — '+TARGET)
        lo=min(z.observed.min(),z.predicted.min());hi=max(z.observed.max(),z.predicted.max())
        fig.add_shape(type='line',x0=lo,y0=lo,x1=hi,y1=hi,line=dict(dash='dash',color='grey'))
        fig.show()
else:
    print('Plots become available after your runs.')

`design.json` records input/code hashes and the exact experiment. `input_audit.csv`, `join_audit.csv` and `exclusions.csv` record coverage. Checkpoints retain inner folds, settings, errors and preprocessing decisions. Complete specifications produce `scores.csv`, `predictions.parquet`, `chosen_hyperparameters.csv`, `retained_features.csv` and `sensitivity_scores.csv`.